In [17]:
from datasets import load_dataset, load_from_disk
import os

In [33]:
PROJ_DIR = '/home/user/WORK/2608_MATS/EN'
DATA_DIR = os.path.join(PROJ_DIR, 'data')
RES_DIR = os.path.join(PROJ_DIR, 'results')
model_path = "/home/user/.cache/huggingface/hub/models--tencent--HY-MT1.5-1.8B/snapshots/dbad03788f49709801014c95d481a514c272ca52"
SYSTEM_PROMPT = "把英语翻译成俄：{text}"
SWITCH_LANG_TRHD = .8


In [18]:
# loading 500 samples for steering vectors calculation
# This data don't use here, load only
steer_data_path = os.path.join(DATA_DIR, './wmt14_ru_en_500_samples')
if os.path.exists(steer_data_path):
    steer_dataset = load_from_disk(steer_data_path)
else:
    steer_dataset = load_dataset("wmt/wmt14", "ru-en", split="train[:500]")
    steer_dataset.save_to_disk(steer_data_path)
steer_dataset

Dataset({
    features: ['translation'],
    num_rows: 500
})

In [22]:
# wmt 14 val part 3000 samples for validation
val_data_path = os.path.join(DATA_DIR, './wmt14_ru_en_validation_full')
if os.path.exists(val_data_path):
    val_dataset = load_from_disk(val_data_path)
else:
    val_dataset = load_dataset("wmt/wmt14", "ru-en", split="validation")
    val_dataset.save_to_disk(val_data_path)
val_dataset

Dataset({
    features: ['translation'],
    num_rows: 3000
})

In [23]:
import re

def is_cyrillic(text: str, threshold: float = 0.7) -> bool:
    """
    Checks whether the text is Russian based on the proportion of Cyrillic characters.

    Args:
        text: Input text
        threshold: Minimum proportion of Cyrillic characters to classify as Russian (0.7 = 70%)

    Returns:
        True if the text is predominantly Russian, False if "съезд"
    """
    if not text or len(text.strip()) == 0:
        return False
    
    # Убираем пробелы, цифры, пунктуацию и эмодзи
    clean_text = re.sub(r'[\s\d\W]', '', text)
    
    if len(clean_text) == 0:
        return True  # Пустой текст или только пробелы/цифры - считаем русским
    
    # Считаем кириллицу (включая Ёё)
    cyrillic_chars = len(re.findall(r'[а-яА-ЯёЁ]', clean_text))
    ratio = cyrillic_chars / len(clean_text)
    
    return ratio >= threshold


is_cyrillic("NVIDIA выпустила новую RTX 4090 👍"), is_cyrillic("NVIDIA выпустила новую RTX 4090 👍", 0.5)

(False, True)

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Путь к модели (используйте реальный путь из вашего кода)
#model_path = "путь_к_вашей_модели"  # например, "./hy_mt_model" или путь к snapshot

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,  # Экономия памяти
    device_map="auto"           # Автоматическое распределение на GPU
)


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'mscale', 'alpha', 'mscale_all_dim', 'beta_fast', 'beta_slow'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'mscale', 'alpha', 'mscale_all_dim', 'beta_fast', 'beta_slow'}
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

In [27]:
# there are some prompt experiments
# english propmpt     'Translate to Russian' don't work

def translate(text, sys_prommpt = '把英语翻译成俄语'):
    prompt = f"{sys_prommpt}: {text}"
    print(prompt)
    print('*****************************')
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            do_sample = False
        )
    
    # drop prompt
    generated = outputs[0][inputs.input_ids.shape[1]:]
    print('len(generated)', len(generated))#, print(generated)
    return tokenizer.decode(generated, skip_special_tokens=True)

print(translate("The quick brown fox jumps over the lazy dog."))

prompts_zh = [
    'Translate to Russian', 
    "把英语翻译成俄",           # Переведи с английского на русский     # 'Translate from english to Russian',
    "请将以下英文翻译成俄语",      # Пожалуйста, переведи следующий английский на русский #     'Please, translate next English  to Russian',
    #"英文：{text}\n俄文：",                # Английский: ... Русский:
    "翻译成俄语",                  # Переведи на русский    'Translate to Russian',
]
for syspr in prompts_zh:
    print(translate("The quick brown fox jumps over the lazy dog.", syspr))
    print('---------------------')

把英语翻译成俄语: The quick brown fox jumps over the lazy dog.
*****************************
len(generated) 27

Быстрый коричневый лис выскакивает через ленивую собаку.
Translate to Russian: The quick brown fox jumps over the lazy dog.
*****************************
len(generated) 134


Активное внимание! Скорописный текст!

Краткое описание: Активное внимание! Скорописный текст!

**Text:** Активное внимание! Скорописный текст!

**Translation to English:** Active attention! Rapid writing text!

**Translation to French:** Attention active ! Texte écrit rapidement !

**Translation to German:** Aktives Aufmerksamkeit! Schneller Schreibtext!

**Translation to Italian:** Attenzione attiva! Testo scritto velocemente!
---------------------
把英语翻译成俄: The quick brown fox jumps over the lazy dog.
*****************************
len(generated) 26


Короткая статья о том, как создать свою собственную монету.
---------------------
请将以下英文翻译成俄语: The quick brown fox jumps over the lazy dog.
**********************

It seems that different system prompts can produce different translation quality, and this could be investigated, but let's skip that for now. I expect that steering will work the same way with different prompts — slightly degrading translation quality and reducing the probability of language switching. How the prompt affects this can be checked later.

In [31]:
import json
import torch
from tqdm import tqdm
from typing import List, Dict, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM

def translate_dataset_no_batch(
    dataset,
    tokenizer,
    model,
    system_prompt: str,
    max_new_tokens_limit: int = 2000,
    source_length_multiplier: float = 2.2,
    output_file: str = "translations.jsonl",
    start_from: int = 0,
    max_examples: Optional[int] = None
):
    """
    Iterates over the dataset one example at a time, dynamically computing max_new_tokens and max_length.
    
    Args:
    dataset: dataset with fields translation.en and translation.ru
    tokenizer: model tokenizer
    model: model for generation
    system_prompt: prompt with the {text} placeholder
    max_new_tokens_limit: maximum allowed number of tokens for generation — maybe tokenize the source first, then compute max_new_tokens
    source_length_multiplier: coefficient for calculating generation length from source length
    output_file: path to save results
    start_from: which example to start from (for resuming)
    max_examples: how many examples to process in total (None = all)
    """
    import os
    
    os.makedirs(os.path.dirname(os.path.abspath(output_file)), exist_ok=True)
    
    total = len(dataset)
    if max_examples:
        total = min(total, max_examples)
        
    print(f"📊 Processing {total} examples one by one")
    print(f"   Maximum token limit: {max_new_tokens_limit}")
    print(f"   Length multiplier: {source_length_multiplier}")    
    
    # some stats
    token_stats = {'min': float('inf'), 'max': 0, 'sum': 0, 'count': 0}
    length_stats = {'min': float('inf'), 'max': 0, 'sum': 0, 'count': 0}
    
    # append mode, if continue 
    mode = 'a' if start_from > 0 else 'w'
    with open(output_file, mode, encoding='utf-8') as f:
        
        # no batches
        for idx in tqdm(range(start_from, total), desc="Перевод"):
            
            source = dataset[idx]['translation']['en']
            reference = dataset[idx]['translation']['ru']
            
            # max_new_tokens, current_max_length dynamic computing - some extra complexity, dropped later
            estimated_source_tokens = len(source.split()) * 1.5
            current_max_new_tokens = int(estimated_source_tokens * source_length_multiplier)
            current_max_new_tokens = min(current_max_new_tokens, max_new_tokens_limit)
            current_max_length = int(current_max_new_tokens / source_length_multiplier) + 50
            
   
            prompt = system_prompt.format(text=source)
            
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=current_max_length
            ).to(model.device)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=current_max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
            
            input_len = inputs.input_ids.shape[1]
            generated_tokens = outputs[0][input_len:]
            prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            
            # save results
            record = {
                "id": idx,
                "source": source,
                "reference": reference,
                "translation": prediction.strip(),
                #"max_new_tokens_used": current_max_new_tokens,
                #"max_length_used": current_max_length,
                #"source_tokens_estimated": int(estimated_source_tokens)
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            #f.flush()

   
#SYSTEM_PROMPT = "请将以下英文翻译成俄语：{text}"
from datetime import datetime
transl_path = os.path.join(RES_DIR, "translations_baseline_3000.jsonl")

print(datetime.now())

#translate_dataset(
translate_dataset_no_batch(
    dataset=val_dataset,
    tokenizer=tokenizer,
    model=model,
    system_prompt=SYSTEM_PROMPT,
    #batch_size=4,
    #max_new_tokens=200, # ?
    output_file=transl_path,
    max_examples=3000  # Для теста — 100 примеров
)
print(datetime.now())


2026-09-13 08:00:28.704421
📊 Processing 3000 examples one by one
   Maximum token limit: 2000
   Length multiplier: 2.2


Перевод:  19%|█████▍                      | 583/3000 [18:02<1:14:47,  1.86s/it]


KeyboardInterrupt: 

In [36]:
import json
import re
from sacrebleu import corpus_bleu, CHRF
from tqdm import tqdm
from typing import List, Dict, Tuple

def is_russian(text: str, threshold: float = SWITCH_LANG_TRHD) -> bool:
    """
    Checks whether the text is Russian based on the proportion of Cyrillic characters.
    """
    if not text or len(text.strip()) == 0:
        return True
    
    # Remove spaces, digits, punctuation, and emoji
    clean_text = re.sub(r'[\s\d\W]', '', text)
    if len(clean_text) == 0:
        return True
    
    cyrillic = len(re.findall(r'[а-яА-ЯёЁ]', clean_text))
    ratio = cyrillic / len(clean_text)
    return ratio >= threshold

def detect_language_switch(text: str) -> bool:
    """
    Returns True if the text is NOT Russian (a switch to another language).
    """
    return not is_russian(text)

def compute_metrics_for_translations(filepath: str) -> Dict:
    """
    Loads a JSONL file with translations and computes metrics.
    
    Returns:
        Dictionary with metrics: average values across all examples
    """
    # Load data
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line))
    
    print(f"📂 Loaded {len(records)} records from {filepath}")
    
    # Extract texts
    sources = [r['source'] for r in records]
    references = [r['reference'] for r in records]
    predictions = [r['translation'] for r in records]
    
    # 1. Compute BLEU
    bleu_score = corpus_bleu(predictions, [references])
    print(f"📊 BLEU: {bleu_score.score:.2f}")
    
    # 2. Compute chrF
    chrf_scorer = CHRF(word_order=2)
    chrf_score = chrf_scorer.corpus_score(predictions, [references])
    print(f"📊 chrF: {chrf_score.score:.2f}")
    
    # 3. Compute language switches
    switch_count = 0
    switch_examples = []
    
    for i, pred in enumerate(predictions):
        if detect_language_switch(pred):
            switch_count += 1
            if len(switch_examples) < 10:  # Keep the first 10 examples for analysis
                switch_examples.append({
                    'id': i,
                    'source': sources[i][:100] + "...",
                    'prediction': pred[:100] + "..."
                })
    
    switch_rate = switch_count / len(predictions) * 100
    print(f"📊 Language switches: {switch_count} / {len(predictions)} ({switch_rate:.2f}%)")
    
    # 4. Length statistics
    src_lengths = [len(s.split()) for s in sources]
    ref_lengths = [len(r.split()) for r in references]
    pred_lengths = [len(p.split()) for p in predictions]
    
    print(f"\n📏 Length statistics (in words):")
    print(f"  Source: avg {sum(src_lengths)/len(src_lengths):.1f}, "
          f"max {max(src_lengths)}, min {min(src_lengths)}")
    print(f"  Reference: avg {sum(ref_lengths)/len(ref_lengths):.1f}, "
          f"max {max(ref_lengths)}, min {min(ref_lengths)}")
    print(f"  Translation: avg {sum(pred_lengths)/len(pred_lengths):.1f}, "
          f"max {max(pred_lengths)}, min {min(pred_lengths)}")
    
    # 5. Switch examples (for manual analysis)
    if switch_examples:
        print(f"\n⚠️ Switch examples (first {len(switch_examples)}):")
        for ex in switch_examples:
            print(f"  [{ex['id']}] Source: {ex['source']}")
            print(f"      Translation: {ex['prediction']}")
            print()
    
    # Return all metrics
    return {
        'total_examples': len(records),
        'bleu': bleu_score.score,
        'chrf': chrf_score.score,
        'switch_count': switch_count,
        'switch_rate': switch_rate,
        'avg_src_len': sum(src_lengths) / len(src_lengths),
        'avg_ref_len': sum(ref_lengths) / len(ref_lengths),
        'avg_pred_len': sum(pred_lengths) / len(pred_lengths),
        'switch_examples': switch_examples
    }
# Load translations
metric_path = os.path.join(RES_DIR, "metrics_baseline.jsonl")

metrics = compute_metrics_for_translations(transl_path)

# Save metrics
import json
with open(metric_path, 'w', encoding='utf-8') as f:
    # Убираем примеры съездов, чтобы файл был компактным
    metrics_clean = {k: v for k, v in metrics.items() if k != 'switch_examples'}
    json.dump(metrics_clean, f, ensure_ascii=False, indent=2)

print(f"Metrics saved to {metric_path}")

📂 Loaded 3000 records from /home/user/WORK/2608_MATS/EN/results/translations_baseline_3000.jsonl
📊 BLEU: 11.50
📊 chrF: 35.25
📊 Language switches: 981 / 3000 (32.70%)

📏 Length statistics (in words):
  Source: avg 18.7, max 82, min 1
  Reference: avg 16.2, max 71, min 1
  Translation: avg 24.3, max 200, min 0

⚠️ Switch examples (first 10):
  [0] Source: A Republican strategy to counter the re-election of Obama...
      Translation: in 2016 is to try to make the election a two-way race.
Республиканская стратегия по противод...

  [6] Source: Unlike in Canada, the American States are responsible for the organisation of federal elections in t...
      Translation: В отличие от Канады, американские штаты несут ответственность за организацию федеральных выборов в С...

  [12] Source: Before the 2006 elections, no US State required voters to show a photo ID card....
      Translation: However, after the election, the US State of Texas decided to require all voters to present a photo ...

  [